# Rift24 — weekend replay

**Paper/backtest only. No live-money trading.** This notebook replays one reconstructed weekend from the frozen fixtures
(real Bitget rToken 15-minute candles + Yahoo daily bars) using `data/demo_state.json`, pre-rendered by the Python engine.
It needs only the standard library. Run from the repository root or from `notebooks/`.

The chronology is enforced by the data layout: everything under `pre` is knowable at decision time; everything under `reveal`
only becomes known at or after the cash open. **Run the cells in order** — the reveal cell is deliberately last.

In [ ]:
import json, datetime as dt
from pathlib import Path
from zoneinfo import ZoneInfo

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
S = json.loads((ROOT / "data" / "demo_state.json").read_text(encoding="utf-8"))
ET = ZoneInfo("America/New_York")
et = lambda iso: dt.datetime.fromisoformat(iso.replace("Z", "+00:00")).astimezone(ET).strftime("%a %d %b %H:%M ET")

R = S["replay"]
lead = S["meta"]["selected_decision_lead_bars"]
T = next(t for t in S["desk"]["times"] if t["lead_bars"] == lead)
print(S["meta"]["paper_only"])
print("data     :", S["meta"]["data"]["label"])
print("provenance:", S["meta"]["data"]["provenance"])
print("session  :", R["session"], "| featured by rule:", S["meta"]["featured_rule"])

## 1 · Friday cash close  →  2 · weekend event

In [ ]:
print("U.S. cash market closed at", et(R["prev_close_utc"]), "- reopens", et(R["open_utc"]))
print()
for e in S["events"]:
    if e["provenance"] == "SOURCED":
        print(f"[SOURCED] {e['headline'][:88]}\n          {e['source']} | maps to: {', '.join(e['mapping']['tickers'])}\n          {e['url']}")
print("\nContext only: not claimed as the cause of any move; timing vs the decision was not verified.")

## 3 · rToken move (what has printed by the decision time)

In [ ]:
print(f"Decision time: {et(R['decision_utc'])}  ({T['label_et']})\n")
print(f"{'symbol':<8}{'void move':>11}{'  last price age':>16}")
for r in T["rows"]:
    p = r["pre"]
    print(f"{p['rtoken']:<8}{p['void_bps']:>+9.0f} bp{p['data_status']['staleness_min']:>13.0f} min")

## 4 · Rift24 decision  (cash still closed; the future is still hidden)

In [ ]:
m = T["model"]
print(f"Historical reference ({m['n']} past large-move sessions, {m['beta']:.2f}x cash-gap response), not a predictive model: implied cash gap = {m['beta']:.2f} x void move\n")
print(f"{'symbol':<8}{'void':>8}{'implied':>9}{'residual':>10}{'cost':>6}{'net':>8}  stance")
for r in T["rows"]:
    p = r["pre"]
    print(f"{p['rtoken']:<8}{p['void_bps']:>+8.0f}{p['implied_cash_bps']:>+9.0f}{p['residual_bps']:>+10.1f}{p['cost_bps']:>6.0f}{p['net_residual_bps']:>+8.1f}  {p['stance']}"
          + (f" ({p['side']})" if p['side'] else f"  [gate {p['binding_gate']}]"))
n_dn = sum(1 for r in T["rows"] if r["pre"]["stance"] == "DO NOTHING")
print(f"\nDO NOTHING on {n_dn} of {len(T['rows'])} names.")

In [ ]:
# Optional: draw the pre-decision paths as an SVG (no third-party libraries).
def svg(paths, w=760, h=300, l=48, r=60, t=12, b=24):
    pts = [(x, y) for p in paths.values() for x, y in p]
    x0, x1 = min(p[0] for p in pts), max(p[0] for p in pts)
    y0, y1 = min(p[1] for p in pts), max(p[1] for p in pts)
    sx = lambda x: l + (x - x0) / (x1 - x0 or 1) * (w - l - r)
    sy = lambda y: h - b - (y - y0) / (y1 - y0 or 1) * (h - b - t)
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {w} {h}" width="{w}" height="{h}" style="background:#0a0d12">']
    out.append(f'<line x1="{l}" x2="{w-r}" y1="{sy(0)}" y2="{sy(0)}" stroke="#4a5568"/>')
    stance = {x["pre"]["rtoken"]: x["pre"]["stance"] for x in T["rows"]}
    for name, p in paths.items():
        c = "#35c27a" if stance[name] != "DO NOTHING" else "#6b7686"
        d = " ".join(f"{'M' if i == 0 else 'L'}{sx(x):.1f},{sy(y):.1f}" for i, (x, y) in enumerate(p))
        out.append(f'<path d="{d}" fill="none" stroke="{c}" stroke-width="1.6"/><text x="{w-r+4}" y="{sy(p[-1][1])+4}" fill="{c}" font-size="11">{name}</text>')
    out.append(f'<text x="{l}" y="{h-6}" fill="#9aa5b6" font-size="11">Fri 16:00 close → decision (bps vs Friday cash close, OBSERVED)</text></svg>')
    return "\n".join(out)

doc = svg(R["pre"]["paths"])
try:
    from IPython.display import SVG, display
    display(SVG(doc))
except Exception:
    (ROOT / "notebooks" / "replay_pre.svg").write_text(doc, encoding="utf-8")
    print("IPython not available - wrote notebooks/replay_pre.svg")

## 5 · Monday cash open  →  6 · actual outcome

Only run this cell after you have looked at the decision above. It reads the `reveal` fields.

In [ ]:
print(f"Cash market opened {et(R['open_utc'])}. Realised outcome:\n")
print(f"{'symbol':<8}{'void':>7}{'cash gap':>10}{'priced-in':>10}  {'autopsy':<15}{'stance':<12}{'net bps':>8}")
tot = []
for r in T["rows"]:
    p, v = r["pre"], r["reveal"]
    ratio = "n/a" if v["priced_in_ratio"] is None else f"{v['priced_in_ratio']:.2f}x"
    net = "" if v["net_bps"] is None else f"{v['net_bps']:+.0f}"
    if v["net_bps"] is not None: tot.append(v["net_bps"])
    print(f"{p['rtoken']:<8}{p['void_bps']:>+7.0f}{v['realized_cash_gap_bps']:>+10.0f}{ratio:>10}  {v['verdict'].split(' (')[0]:<15}{p['stance']:<12}{net:>8}")
print()
print(f"{len(tot)} trade(s), mean {sum(tot)/len(tot):+.0f} bps net (flattened just before the open)." if tot else "No trades.")
print("\nIs this weekend typical? Exploratory-strategy day return on book, all weekends:")
for w in R["weekends"]:
    mark = " <-- this one (chosen by rule: most recent)" if w["open_date"] == R["session"] else ""
    print(f"  {w['open_date']}  trades={w['rift24_trades']}  {w['rift24_day_return_pct']:+.2f}%{mark}")